In [ ]:
import pandas as pd
import plotly.graph_objects as go
from prophet import Prophet

In [ ]:
# 1. Accumulate and Preprocess Data
# The dataset contains columns: Province/State, Country/Region, Lat, Long, Date, Confirmed, Deaths, Recovered, Active, WHO Region
df = pd.read_csv('/content/covid_19_clean_complete.csv')
df['Date'] = pd.to_datetime(df['Date'])

In [ ]:
df

,Province/State,Country/Region,Lat,Long,Date,Confirmed,Deaths,Recovered,Active,WHO Region
0,NaN,Afghanistan,33.939110,67.709953,2020-01-22,0,0,0,0,Eastern Mediterranean
1,NaN,Albania,41.153300,20.168300,2020-01-22,0,0,0,0,Europe
2,NaN,Algeria,28.033900,1.659600,2020-01-22,0,0,0,0,Africa
3,NaN,Andorra,42.506300,1.521800,2020-01-22,0,0,0,0,Europe
4,NaN,Angola,-11.202700,17.873900,2020-01-22,0,0,0,0,Africa
...,...,...,...,...,...,...,...,...,...,...
49063,NaN,Sao Tome and Principe,0.186400,6.613100,2020-07-27,865,14,734,117,Africa
49064,NaN,Yemen,15.552727,48.516388,2020-07-27,1691,483,833,375,Eastern Mediterranean
49065,NaN,Comoros,-11.645500,43.333300,2020-07-27,354,7,328,19,Africa
49066,NaN,Tajikistan,38.861000,71.276100,2020-07-27,7235,60,6028,1147,Europe


In [ ]:
df['Date']

,Date
0,2020-01-22
1,2020-01-22
2,2020-01-22
3,2020-01-22
4,2020-01-22
...,...
49063,2020-07-27
49064,2020-07-27
49065,2020-07-27
49066,2020-07-27


In [ ]:
# Group by Date to analyze global trends
global_df = df.groupby('Date')[['Confirmed', 'Recovered', 'Deaths']].sum().reset_index()
global_df

,Date,Confirmed,Recovered,Deaths
0,2020-01-22,555,28,17
1,2020-01-23,654,30,18
2,2020-01-24,941,36,26
3,2020-01-25,1434,39,42
4,2020-01-26,2118,52,56
...,...,...,...,...
183,2020-07-23,15510481,8710969,633506
184,2020-07-24,15791645,8939705,639650
185,2020-07-25,16047190,9158743,644517
186,2020-07-26,16251796,9293464,648621


In [ ]:
# Analyze Trend: Infection and Recovery Rates
# Calculate daily new cases (rate of infection/recovery)
global_df['Infection_Rate'] = global_df['Confirmed'].diff().fillna(0)
global_df['Recovery_Rate'] = global_df['Recovered'].diff().fillna(0)

In [ ]:
global_df

,Date,Confirmed,Recovered,Deaths,Infection_Rate,Recovery_Rate
0,2020-01-22,555,28,17,0.0,0.0
1,2020-01-23,654,30,18,99.0,2.0
2,2020-01-24,941,36,26,287.0,6.0
3,2020-01-25,1434,39,42,493.0,3.0
4,2020-01-26,2118,52,56,684.0,13.0
...,...,...,...,...,...,...
183,2020-07-23,15510481,8710969,633506,282756.0,169714.0
184,2020-07-24,15791645,8939705,639650,281164.0,228736.0
185,2020-07-25,16047190,9158743,644517,255545.0,219038.0
186,2020-07-26,16251796,9293464,648621,204606.0,134721.0


In [ ]:
# Visualize Daily Trends using Plotly
fig_rates = go.Figure()
fig_rates.add_trace(go.Scatter(x=global_df['Date'], y=global_df['Infection_Rate'],
                               mode='lines', name='Daily New Infections'))
fig_rates.add_trace(go.Scatter(x=global_df['Date'], y=global_df['Recovery_Rate'],
                               mode='lines', name='Daily New Recoveries', line=dict(color='green')))

fig_rates.update_layout(title='COVID-19 Global Daily Infection vs Recovery Rate',
                        xaxis_title='Date', yaxis_title='Number of Cases',
                        hovermode='x unified')
fig_rates.show()

In [ ]:
#Time Series Prediction with Facebook Prophet
# Prophet requires a dataframe with columns 'ds' (datestamp) and 'y' (value)
df_prophet = global_df[['Date', 'Confirmed']].rename(columns={'Date': 'ds', 'Confirmed': 'y'})

model = Prophet(interval_width=0.95) # 95% confidence interval
model.fit(df_prophet)

INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


In [ ]:
# Predict 7 days into the future
future = model.make_future_dataframe(periods=7)
forecast = model.predict(future)

In [ ]:
future

,ds
0,2020-01-22
1,2020-01-23
2,2020-01-24
3,2020-01-25
4,2020-01-26
...,...
190,2020-07-30
191,2020-07-31
192,2020-08-01
193,2020-08-02


In [ ]:
forecast

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,additive_terms,additive_terms_lower,additive_terms_upper,weekly,weekly_lower,weekly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2020-01-22,-9.613288e+03,-1.889596e+05,1.330417e+05,-9.613288e+03,-9.613288e+03,-11063.558307,-11063.558307,-11063.558307,-11063.558307,-11063.558307,-11063.558307,0.0,0.0,0.0,-2.067685e+04
1,2020-01-23,-6.933409e+03,-1.637161e+05,1.437875e+05,-6.933409e+03,-6.933409e+03,-1117.543863,-1117.543863,-1117.543863,-1117.543863,-1117.543863,-1117.543863,0.0,0.0,0.0,-8.050953e+03
2,2020-01-24,-4.253530e+03,-1.567919e+05,1.641826e+05,-4.253530e+03,-4.253530e+03,10080.978737,10080.978737,10080.978737,10080.978737,10080.978737,10080.978737,0.0,0.0,0.0,5.827449e+03
3,2020-01-25,-1.573651e+03,-1.591556e+05,1.701295e+05,-1.573651e+03,-1.573651e+03,13750.326871,13750.326871,13750.326871,13750.326871,13750.326871,13750.326871,0.0,0.0,0.0,1.217668e+04
4,2020-01-26,1.106228e+03,-1.492589e+05,1.677131e+05,1.106228e+03,1.106228e+03,7298.791978,7298.791978,7298.791978,7298.791978,7298.791978,7298.791978,0.0,0.0,0.0,8.405020e+03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,2020-07-30,1.674503e+07,1.657021e+07,1.689742e+07,1.671786e+07,1.677114e+07,-1117.543863,-1117.543863,-1117.543863,-1117.543863,-1117.543863,-1117.543863,0.0,0.0,0.0,1.674392e+07
191,2020-07-31,1.694902e+07,1.678695e+07,1.712553e+07,1.690749e+07,1.698977e+07,10080.978737,10080.978737,10080.978737,10080.978737,10080.978737,10080.978737,0.0,0.0,0.0,1.695911e+07
192,2020-08-01,1.715301e+07,1.700725e+07,1.733601e+07,1.709304e+07,1.720615e+07,13750.326871,13750.326871,13750.326871,13750.326871,13750.326871,13750.326871,0.0,0.0,0.0,1.716677e+07
193,2020-08-02,1.735701e+07,1.719284e+07,1.754462e+07,1.727597e+07,1.742978e+07,7298.791978,7298.791978,7298.791978,7298.791978,7298.791978,7298.791978,0.0,0.0,0.0,1.736430e+07


In [ ]:
# 4. Visualize the Prediction
fig_pred = go.Figure()

# Plot actual data
fig_pred.add_trace(go.Scatter(x=df_prophet['ds'], y=df_prophet['y'],
                             mode='markers', name='Actual Confirmed Cases'))
# Plot predicted trend
fig_pred.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat'],
                             mode='lines', name='Predicted Forecast', line=dict(color='red')))
# Plot Uncertainty Interval
fig_pred.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat_upper'],
                             fill='tonexty', mode='none', name='Upper Bound (CI)',
                             fillcolor='rgba(255, 0, 0, 0.2)'))

fig_pred.update_layout(title='COVID-19 7-Day Prediction: Confirmed Cases',
                       xaxis_title='Date', yaxis_title='Total Cases')
fig_pred.show()

In [ ]:
#Output
next_week_forecast = forecast.iloc[-1]['yhat']
print(f"Predicted total cases in one week: {int(next_week_forecast)}")

Predicted total cases in one week: 17558893
